# Case Study 3: Market Making the SPY Limit Order Book

**Name:** _your name_  **Computing ID:** _your id_

**Head trader's five questions**
1. How much do fees and rebates matter?
2. How much does latency cost?
3. How important is queue priority?
4. Is inventory skew the best way to tilt a quote?
5. What does P&L look like *after* markout (adverse selection)?

**Submission checklist**
- [ ] Notebook runs top-to-bottom without error
- [ ] Tape read from `TAPE_PATH = "data/20260812_spy.txt"` (zip includes `data/` next to the notebook)
- [ ] Every plot has title, axis labels, legend
- [ ] Same evaluation window (`MAX_MESSAGES`) used for every comparison
- [ ] Prose interpretation under every section

## Setup

In [ ]:
import sys, copy, re
from dataclasses import dataclass
from datetime import datetime, time as clock_time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# fake_broker.py / main.py are provided in the grading environment; adjust path if needed
sys.path.append("lectures/module08/backtest")
from fake_broker import FakeBroker, QuoteUpdate, TradePrint, OrderFilled, OrderAccepted, OrderCanceled
import main as baseline_main   # provided main.py: handle_next_tick, run_backtest, constants
from main import QUOTE_OFFSET, ORDER_SIZE, INVENTORY_SKEW, MAX_INVENTORY

TAPE_PATH = "data/20260812_spy.txt"   # NOT ../../../data/...
MAX_MESSAGES = 300_000                # FIXED evaluation window for every comparison
MARKET_OPEN, MARKET_CLOSE = clock_time(9, 30), clock_time(16, 0)

## Section 0 — What's Already Built

Summarize in your own words (3-5 sentences) how `FakeBroker` and `handle_next_tick` work, and note the baseline's implicit assumptions:
zero latency, front-of-queue fills, no fees, quotes cancelled/re-sent on every tick.

In [ ]:
def new_book():
    # same fields as main.run_backtest's book, plus extras for this case study
    return {"time": None, "best_bid": None, "best_ask": None, "last_trade_price": None,
            "inventory": 0, "cash": 0.0, "pnl_history": [],
            "fills": [], "rebates": 0.0, "fees": 0.0,   # TODO: extra fields (mid history, ...)
            }

# 1) Baseline exactly as provided (zero latency, optimistic queue, no fees). Reuse for all comparisons.
baseline_book = baseline_main.run_backtest(TAPE_PATH, plot=True, max_messages=MAX_MESSAGES)

# 2) Your configurable version. main.handle_next_tick reads module constants and has no hook for
#    fees, latency or alternative skews, so copy it and take a cfg dict instead, e.g.
#    def handle_next_tick(broker, book, cfg): ...
def run_strategy(cfg=None, broker_cls=FakeBroker, max_messages=MAX_MESSAGES):
    """Mirror of main.run_backtest, but returns a results dict (book, fills, pnl series, inventory series)."""
    raise NotImplementedError

# Sanity check: run_strategy(default cfg) should reproduce baseline_book's final PnL

## Section 1 — Calibrating the Model to Real Data  (10 pts)

**To do**
- [ ] Estimate $\sigma$ from midprice realized volatility (state units; stay consistent)
- [ ] Estimate $k, A$ for $\lambda(\delta)=Ae^{-k\delta}$: bucket trades by distance from prevailing mid, fit exponential to counts (prose: approach + justification)
- [ ] Run `simulate_as` with calibrated params vs. the real `main.run_backtest` baseline (`baseline_book`) over the same session
- [ ] Paragraph: compare shape of PnL, inventory, drawdown; explain divergences

In [ ]:
def load_tape(path=TAPE_PATH, max_messages=MAX_MESSAGES):
    """TODO: parse quotes (time, bid, ask, sizes) and trades (time, price, size, exchange) into DataFrames.
    Note: fake_broker's parser drops sizes/exchange - you may need your own parser for bid_size/ask_size."""
    raise NotImplementedError

def estimate_sigma(quotes):
    raise NotImplementedError

def fit_intensity(trades, quotes):
    """Return (A, k)."""
    raise NotImplementedError

# paste simulate_as from Module 8 slides here, then run with calibrated params

**Interpretation (Section 1):** _TODO_

## Section 2 — Fees and Rebates  (15 pts)

**To do**
- [ ] Add per-share maker rebate $\rho$ on every `OrderFilled` (broker or accounting layer)
- [ ] Report total rebate revenue and its fraction of gross P&L at $\rho \approx \$0.0020$–$0.0028$
- [ ] Sweep $\rho$: zero, realistic range, negative (inverted venue); find break-even $\rho$
- [ ] End-of-day flatten by crossing the spread with taker fee ≈ $0.0030/share; how much of rebate revenue does it eat?
- [ ] Plot: net P&L vs. $\rho$
- [ ] Paragraph answering: how much do fees/rebates matter?

In [ ]:
RHO_GRID = [-0.0020, -0.0010, 0.0, 0.0010, 0.0020, 0.0025, 0.0028]
TAKER_FEE = 0.0030

def rebate_sweep(rho_grid):
    raise NotImplementedError

# fig: net PnL vs rho (title, axis labels, legend)

**Interpretation (Section 2):** _TODO_

## Section 3 — Latency  (15 pts)

**To do**
- [ ] Extend broker: outbound latency (buffer `send_limit_order`/`cancel_order`, apply $L$ ms of tape-time later)
- [ ] Market-data latency (delay what `poll()` reveals by $L$ ms)
- [ ] Document what happens to a stale resting order the market has moved past before the cancel lands
- [ ] Run $L \in \{0,1,5,20,100,500\}$ ms
- [ ] Plot net P&L, fill count, and realized spread capture (markout-free) vs. $L$
- [ ] Find the latency where strategy stops being profitable; connect to Module 8 latency table

In [ ]:
class LatencyBroker(FakeBroker):
    """TODO: add latency_ms; queue outgoing orders/cancels with an activation tape-time;
    optionally delay market-data visibility."""
    def __init__(self, data_path, latency_ms=0):
        super().__init__(data_path)
        self.latency_ms = latency_ms
        # TODO: pending-action queue

LATENCY_GRID_MS = [0, 1, 5, 20, 100, 500]
# results_lat = {L: run_strategy(cfg, broker_cls=lambda p: LatencyBroker(p, L)) for L in LATENCY_GRID_MS}

**Interpretation (Section 3):** _TODO_

## Section 4 — Queue Priority  (15 pts)

**To do**
- [ ] Pessimistic fill model: join back of queue behind displayed `bid_size`/`ask_size` at your price
- [ ] Deplete queue ahead using trade prints and quote-size changes (document simplifying assumptions; say explicitly that Level 1 can't reconstruct queue position)
- [ ] Fill only after queue ahead is exhausted
- [ ] Compare optimistic vs. pessimistic (fees, latency, quoting fixed): fill count, fill rate vs. quoted distance, net P&L (plots)
- [ ] Paragraph: how much profit was an artifact of perfect priority? Best estimate of truth between the bounds

In [ ]:
class QueueAwareBroker(FakeBroker):
    """TODO: parse bid_size/ask_size (fake_broker's parser discards them), track queue_ahead per resting order."""
    pass

# results_fill = {"optimistic": ..., "pessimistic": ...}

**Assumptions & interpretation (Section 4):** _TODO_

## Section 5 — Alternative Quoting Strategies  (15 pts)

**To do** (at least one alternative)
- [ ] Order-flow imbalance skew, **or** volatility-adaptive spread, **or** quote-depth skew
- [ ] Compare vs. baseline linear inventory skew and naive symmetric (no skew); fees/latency/fill model fixed
- [ ] Report net P&L, inventory path + mean/max |inventory|, fill count for all three
- [ ] Check: no lookahead (only use ticks already read)
- [ ] Did it help, hurt, or just trade P&L for inventory risk?

In [ ]:
def make_strategy(kind):
    """kind in {"symmetric", "inventory_skew", "ofi_skew" / "vol_adaptive" / "depth_skew"}.
    Return a handle_next_tick-style function."""
    raise NotImplementedError

# results_strat = {k: run_strategy(cfg_k) for k in [...]}
# summary table (DataFrame) + inventory time series plot

**Interpretation (Section 5):** _TODO_

## Section 6 — Markout: Are You Getting Picked Off?  (15 pts)

**Sign convention (state explicitly):** _TODO, e.g. markout is signed so positive = price moved in our favor: buy: mid(t+h) − fill; sell: fill − mid(t+h)._

**To do**
- [ ] Markout at h = 1s, 5s, 30s, 5min for every fill
- [ ] Markout curve (mean vs. horizon) with confidence band (SE or bootstrap)
- [ ] Compare ≥2 variants (optimistic vs. pessimistic fills, or baseline vs. alternative skew; or small vs. large `QUOTE_OFFSET`)
- [ ] Does markout stabilize at some horizon?
- [ ] Fill in the edge equation with your own numbers

In [ ]:
HORIZONS_S = [1, 5, 30, 300]

def compute_markouts(fills, mid_series, horizons=HORIZONS_S):
    """fills: DataFrame(time, side, price, size); mid_series: Series indexed by time.
    Use asof lookup of mid at t+h. Fills too close to the end of the tape: drop & note it."""
    raise NotImplementedError

def markout_curve(markouts_df, n_boot=1000):
    raise NotImplementedError

# edge table
edge = pd.Series({"captured spread": np.nan, "rebates": np.nan, "fees": np.nan, "adverse selection": np.nan})

**Interpretation (Section 6):** _TODO — how much gross spread survives adverse selection?_

## Section 7 — Full Backtest, Tearsheet, Interpretation  (15 pts)

**Chosen configuration:** fee/rebate = _TODO_; latency = _TODO_; fill model = _TODO_; quoting strategy = _TODO_.

**Rolling recalibration:** refit σ, k, A (and signal params) every _N_ minutes using only past data. Justification: _TODO_.

In [ ]:
REFIT_EVERY_MIN = 30

def rolling_recalibration(book, now):  # TODO
    raise NotImplementedError

# final = run_strategy(final_cfg)
tearsheet = pd.DataFrame({"Metric": [
    "Gross P&L", "Rebates earned", "Fees paid", "Net P&L",
    "Estimated adverse selection cost (markout)", "Fill count (buy / sell)",
    "Mean |inventory|", "Max |inventory|", "End-of-day inventory",
    "PnL volatility (intraday)", "Max intraday drawdown (PnL)"], "Value": np.nan})
tearsheet

In [ ]:
# Plot: final strategy vs. (a) main.py baseline as-is (zero latency, optimistic queue, no fees) vs. (b) naive symmetric

### Interpretation
- **Ranking of frictions** (fees, latency, queue priority, adverse selection) by cut into gross P&L: _TODO_
- **Did the Section 5 signal survive realistic frictions?** _TODO_
- **First change I'd make to the book** (cite tearsheet/markout numbers): _TODO_

### Desk recommendation (3–5 sentences to the head trader)
_TODO_

## Section 8 — Bonus (optional, up to +5)

Pick one: hard inventory kill-switch · toxicity-aware quoting · fitted fill-probability model.
Explain your choice and what it revealed vs. Section 7.

In [ ]:
# TODO (optional)